# Basic EDA - Time Series

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

## 1. Import Data

In [ ]:
df = pd.read_csv('data/timeseries.csv', parse_dates=['date'], index_col='date')
df = df.asfreq('D')  # ensure daily frequency
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.describe()

In [ ]:
df.plot(title='Daily Unit Sales')
plt.show()

## 2. Outlier Detection (IQR)

In [ ]:
Q1 = df['unit_sales'].quantile(0.25)
Q3 = df['unit_sales'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = df[(df['unit_sales'] < lower) | (df['unit_sales'] > upper)]
print(f'Outliers detected: {len(outliers)}')
print(f'Bounds: [{lower:.1f}, {upper:.1f}]')
outliers

In [ ]:
# Cap outliers at bounds
df['unit_sales'] = df['unit_sales'].clip(lower, upper)
print('Outliers capped.')

## 3. Handling Missing Values

In [ ]:
missing = df.isnull().sum()
print(f'Missing values:\n{missing}')

In [ ]:
# Forward fill then backward fill any remaining
df['unit_sales'] = df['unit_sales'].ffill().bfill()
print(f'Missing after fill: {df.isnull().sum().values[0]}')

## 4. Seasonal Decomposition

In [ ]:
decomp = seasonal_decompose(df['unit_sales'], model='additive', period=7)
fig = decomp.plot()
fig.set_size_inches(12, 8)
plt.show()

## 5. Stationarity Analysis (ADF Test)

In [ ]:
result = adfuller(df['unit_sales'].dropna())
print(f'ADF Statistic: {result[0]:.4f}')
print(f'p-value: {result[1]:.4f}')
print(f'Critical values:')
for key, val in result[4].items():
    print(f'  {key}: {val:.4f}')
if result[1] < 0.05:
    print('=> Series is stationary (reject H0)')
else:
    print('=> Series is non-stationary (fail to reject H0)')

## 6. Autocorrelation (ACF & PACF)

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))
plot_acf(df['unit_sales'], lags=30, ax=ax1)
plot_pacf(df['unit_sales'], lags=30, ax=ax2)
plt.show()

## 7. Save Cleaned Data

In [ ]:
df.to_csv('data/cleaned_timeseries.csv')
print('Cleaned data saved to data/cleaned_timeseries.csv')
df.head()